# LLM Customs Anomaly Explainability

This notebook sends flagged transactions (`low`, `medium`, `high`) to an LLM and saves a clean output with:
- `llm_explanation`
- `llm_issue_type`
- `llm_validation`
- `llm_confidence`
- `llm_next_step`

Normal rows are skipped entirely — the LLM only explains genuine flags.

## 1) Configuration

This is the only cell you need to edit before running the notebook.
- Set `PROVIDER` to `"openai"` or `"deepseek"`
- Fill in your API keys
- Set `MAX_ROWS` to a small number for testing, or `None` for a full run

In [1]:
# Parameters (injected by papermill)
input_path  = r"C:\Users\fawaz\Desktop\Project\outputs\master_anomalies.csv"
output_path = r"C:\Users\fawaz\Desktop\Project\outputs\LLM_Explainability.csv"
year        = "2024"


In [ ]:
# PROVIDER — uncomment the one you want to use
PROVIDER = "openai"
#PROVIDER = "deepseek"

# API KEY LOADING from .env
import os
from pathlib import Path
from dotenv import load_dotenv

env_path = Path(r"C:\Users\fawaz\Desktop\Project\.env")
load_dotenv(dotenv_path=env_path, override=True)

OPENAI_API_KEY   = os.getenv("OPENAI_API_KEY")
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")

if PROVIDER == "openai" and not OPENAI_API_KEY:
    raise RuntimeError(f"OPENAI_API_KEY not found in {env_path}")
if PROVIDER == "deepseek" and not DEEPSEEK_API_KEY:
    raise RuntimeError(f"DEEPSEEK_API_KEY not found in {env_path}")

# MODELS & ENDPOINTS
OPENAI_MODEL     = "gpt-4o-mini"          # or "gpt-4o"
OPENAI_BASE_URL  = "https://api.openai.com/v1"

DEEPSEEK_MODEL    = "deepseek-chat"        # or "deepseek-reasoner"
DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"

# RUN SETTINGS
MAX_ROWS        = None # set to None for full run
SLEEP_SECONDS   = 0.5
TIMEOUT_SECONDS = 120
TEMPERATURE     = 0.2

print(f"Loaded keys from {env_path}")
print(f"Active provider: {PROVIDER}")

Loaded keys from C:\Users\fawaz\Desktop\Project\.env
Active provider: openai


## 2) Imports & provider setup

In [3]:
import os
import re
import json
import time
from typing import Any, Dict
import pandas as pd
import requests
from tqdm.auto import tqdm

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 120)

# Build provider config from Step 1 values
PROVIDER_CONFIG = {
    "openai": {
        "model":    OPENAI_MODEL,
        "base_url": OPENAI_BASE_URL,
        "api_key":  OPENAI_API_KEY,
    },
    "deepseek": {
        "model":    DEEPSEEK_MODEL,
        "base_url": DEEPSEEK_BASE_URL,
        "api_key":  DEEPSEEK_API_KEY,
    },
}

if PROVIDER not in PROVIDER_CONFIG:
    raise ValueError(f"Unsupported PROVIDER: '{PROVIDER}'. Use 'openai' or 'deepseek'.")

MODEL_NAME = PROVIDER_CONFIG[PROVIDER]["model"]
BASE_URL   = PROVIDER_CONFIG[PROVIDER]["base_url"]
API_KEY    = PROVIDER_CONFIG[PROVIDER]["api_key"]

if not API_KEY or API_KEY.startswith("YOUR_"):
    raise ValueError(f"Missing API key for provider: {PROVIDER}")

print(f"Provider : {PROVIDER}")
print(f"Model    : {MODEL_NAME}")
print(f"Max rows : {MAX_ROWS if MAX_ROWS else 'ALL'}")

Provider : openai
Model    : gpt-4o-mini
Max rows : ALL


## 3) Load data and filter to flagged rows only

Only `low`, `medium`, and `high` rows are sent to the LLM.


In [4]:
df = pd.read_csv(input_path, low_memory=False)

print("Full dataset shape:", df.shape)
print("\nLevel distribution:")
print(df['final_level'].value_counts(dropna=False))

FLAG_KEEP = ["LOW", "MEDIUM", "HIGH"]

flagged_df = df[
    df["final_level"].astype(str).str.strip().str.upper().isin(FLAG_KEEP)
].copy().reset_index(drop=True)

print(f"\nFlagged rows (LOW/MEDIUM/HIGH): {len(flagged_df)}")

# Apply MAX_ROWS limit for testing
work_df = flagged_df.head(MAX_ROWS).copy() if MAX_ROWS else flagged_df.copy()
work_df = work_df.reset_index(drop=True)

print(f"Rows to process: {len(work_df)}")
if MAX_ROWS:
    print(f"[TEST MODE — limited to {MAX_ROWS} rows]")


Full dataset shape: (1000, 78)

Level distribution:
final_level
NORMAL    510
LOW       299
MEDIUM     96
HIGH       95
Name: count, dtype: int64

Flagged rows (LOW/MEDIUM/HIGH): 490
Rows to process: 490


## 4) System prompt and user prompt builder

The system prompt defines the analyst role, detection logic, and strict JSON output format.
The user prompt is built per row using `safe_get` so missing fields never cause crashes.

In [5]:
SYSTEM_PROMPT = """
You are a senior customs analyst at the Bahrain Ministry of Finance and National Economy. Your role is to review transactions that an automated anomaly detection pipeline has flagged, and decide whether each flag reflects a genuine risk or a false positive.

You will receive ONE flagged customs transaction at a time. Use the transaction details and the anomaly signals together — neither alone is sufficient. A high anomaly score on a transaction whose details make sense in context is a likely false positive; a moderate score on a transaction with a suspicious field combination may be a genuine risk.

=== HOW TO READ THE FIELDS ===

TRANSACTION DETAILS
- Transaction Date: When the declaration was filed. Interpret prices and patterns in the context of their year.
- Trader CR: The Commercial Registration of the Bahrain-side company.
- Trade Direction: Import / Export / Re-export.
- HS Code: Harmonized System product code. First 2 digits = chapter, first 6 = international standard.
- Declared Goods Description: Official HS description.
- Commercial Description: Trader free-text. Mismatches with HS description signal misclassification.
- Country of Origin / Export / Destination: Origin = production country. Export = shipping country. Destination = receiving country.
- Unit of Measure (UOM): KGM = kilogram, LTR = litre, NMB = number/count.
- Declared Quantity, Unit Price (BHD), Price Basis (Incoterm), Invoice Total, Local Total, Net/Gross Weight.

ANOMALY SIGNALS (three independent dimensions)
1. PRICE DIMENSION:
   - Robust Z-score: MADs from the same-HS same-year market median. |z| > 2.5 = anomalous. Negative = under-invoicing, positive = over-invoicing.
   - Price Level: NORMAL / LOW / MEDIUM / HIGH — the pipeline severity for this dimension.

2. TRADE PATTERN DIMENSION:
   - Trade Pattern Score: Composite of partner rarity, route rarity, and burstiness (0-100).
   - Partner Rarity Score: How rare this trading partner is for this HS code (0-100).
   - Route Rarity Score: How rare this origin-destination route is (0-100).
   - Burstiness Score: Detects sudden volume spikes for this HS (0-100).
   - Pattern Level: NORMAL / LOW / MEDIUM / HIGH.

3. CR PROFILE DIMENSION (trader behaviour vs own history):
   - CR Price Severity: How unusual this price is vs the trader own past prices (0-100).
   - CR Burst Severity: Volume spike vs the trader own baseline (0-100).
   - CR Portfolio Distance: How far this product is from the trader usual product mix (0-100).
   - CR Partner Rarity: How unusual this partner is for this trader (0-100).
   - CR Profile Score: Weighted composite of the above (0-100).
   - CR Profile Level: NORMAL / LOW / MEDIUM / HIGH.
   - First-time flags: Whether this is the first time this trader has used this product/partner/combination.

COMPOSITE (final level):
- Final Level: Agreement-based rule. HIGH requires strong evidence from any dimension. MEDIUM requires two corroborating signals. LOW requires at least two weak signals agreeing. A single weak signal alone = NORMAL.
- Final Reason: Which dimensions drove the final level.

=== YOUR TASK ===

Return ONLY a single valid JSON object — no markdown, no preamble, no explanation outside the JSON. Use exactly these fields:

{
  "explanation": "2-4 sentences explaining what is anomalous or why the flag is likely a false positive. Reference specific fields and the date context.",
  "issue_type": "One of: under_invoicing | over_invoicing | misclassification | unusual_partner | unusual_route | volume_spike | re_export_anomaly | cr_profile_deviation | benign_outlier | other",
  "validation": "TRUE_POSITIVE or FALSE_POSITIVE",
  "confidence": "low | medium | high",
  "next_step": "One concrete action a customs officer should take.",
  "validation_reasoning": "1-2 sentences justifying your TRUE_POSITIVE / FALSE_POSITIVE decision."
}
"""


def compact_text(text: str) -> str:
    return re.sub(r"\s+", " ", str(text)).strip()


def safe_get(row: pd.Series, col: str, default: str = "Not provided") -> str:
    if col not in row.index:
        return default
    val = row[col]
    if pd.isna(val):
        return default
    return compact_text(str(val))


def safe_date(row: pd.Series, col: str, default: str = "Not provided") -> str:
    if col not in row.index:
        return default
    val = row[col]
    if pd.isna(val):
        return default
    try:
        return pd.to_datetime(val).strftime("%Y-%m-%d")
    except Exception:
        return compact_text(str(val))


def build_user_prompt(row: pd.Series) -> str:
    return f"""Analyze this flagged customs transaction:

--- TRANSACTION ---
Transaction Date:           {safe_date(row, 'declaration_date')}
Trader CR:                  {safe_get(row, 'active_cr')}
Trade Direction:            {safe_get(row, 'trade_type')}
HS Code:                    {safe_get(row, 'hs_code')} (HS6: {safe_get(row, 'hs6')}, Chapter: {safe_get(row, 'hs2')})
Declared Goods Description: {safe_get(row, 'hs_desc')}
Commercial Description:     {safe_get(row, 'commercial_description')}
Country of Origin:          {safe_get(row, 'country_of_origin')} ({safe_get(row, 'country_of_origin_code')})
Country of Export:          {safe_get(row, 'country_of_export')} ({safe_get(row, 'country_of_export_code')})
Country of Destination:     {safe_get(row, 'country_of_destination')} ({safe_get(row, 'country_of_destination_code')})
Unit of Measure:            {safe_get(row, 'uom')}
Declared Quantity:          {safe_get(row, 'qty_by_uom')}
Declared Unit Price (BHD):  {safe_get(row, 'actual_unit_price')}
Price Basis:                {safe_get(row, 'price_basis')}
Invoice Total:              {safe_get(row, 'invoice_amount_clean')}
Local Total (BHD):          {safe_get(row, 'local_amount_clean')}
Net Weight (kg):            {safe_get(row, 'net_weight_clean')}
Gross Weight (kg):          {safe_get(row, 'gross_weight_clean')}

--- PRICE ANOMALY ---
Robust Z-score:             {safe_get(row, 'robust_z')}
Peer Median Price:          {safe_get(row, 'median')}
Price Level:                {safe_get(row, 'price_level')}

--- TRADE PATTERN ---
Trade Pattern Score:        {safe_get(row, 'trade_pattern_score')}
Partner Rarity Score:       {safe_get(row, 'partner_rarity_score')}
Route Rarity Score:         {safe_get(row, 'route_rarity_score')}
Burstiness Score:           {safe_get(row, 'burstiness_score')}
Pattern Level:              {safe_get(row, 'pattern_level')}

--- CR PROFILE (trader behaviour vs own history) ---
CR Price Severity:          {safe_get(row, 'cr_price_severity')}
CR Burst Severity:          {safe_get(row, 'cr_burst_severity')}
CR Portfolio Distance:      {safe_get(row, 'cr_portfolio_distance')}
CR Portfolio Severity:      {safe_get(row, 'cr_portfolio_severity')}
CR Partner Rarity:          {safe_get(row, 'cr_partner_rarity')}
First-time Product:         {safe_get(row, 'cr_hs_first_time_in_window')}
First-time Partner:         {safe_get(row, 'cr_partner_first_time')}
First-time Product+Partner: {safe_get(row, 'cr_partner_hs_first_time')}
CR Profile Score:           {safe_get(row, 'cr_profile_score')}
CR Profile Level:           {safe_get(row, 'cr_profile_level')}

--- COMPOSITE ---
Final Level:                {safe_get(row, 'final_level')}
Final Reason:               {safe_get(row, 'final_reason')}
"""


## 5) LLM client and JSON parser

Uses a raw `requests` call so it works identically for OpenAI and DeepSeek.
The JSON parser tries direct parse first, then falls back to regex extraction
so minor model formatting quirks never crash the loop.

In [6]:
class LLMClient:
    def __init__(self, provider, model_name, api_key, base_url, temperature=0.2, timeout=120):
        self.provider    = provider
        self.model       = model_name
        self.api_key     = api_key
        self.base_url    = base_url
        self.temperature = temperature
        self.timeout     = timeout

    def generate(self, system_prompt: str, user_prompt: str) -> str:
        url     = f"{self.base_url.rstrip('/')}/chat/completions"
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type":  "application/json",
        }
        payload = {
            "model": self.model,
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": user_prompt},
            ],
            "temperature": self.temperature,
        }
        response = requests.post(url, headers=headers, json=payload, timeout=self.timeout)
        response.raise_for_status()
        return response.json()["choices"][0]["message"]["content"]


EXPECTED_KEYS = ["explanation", "issue_type", "validation", "validation_reasoning", "confidence", "next_step"]


def extract_json_object(text: str) -> dict:
    if not isinstance(text, str):
        raise ValueError("Model response is not text.")
    text = text.strip()
    # Strip markdown fences if present
    text = re.sub(r"^```json\s*", "", text)
    text = re.sub(r"^```\s*",     "", text)
    text = re.sub(r"\s*```$",     "", text)
    # Try direct parse
    try:
        obj = json.loads(text)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass
    # Fallback: find first {...} block
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if match:
        try:
            obj = json.loads(match.group(0))
            if isinstance(obj, dict):
                return obj
        except Exception:
            pass
    raise ValueError("Model response was not valid JSON.")


def normalize_result(obj: Dict[str, Any]) -> Dict[str, str]:
    return {key: compact_text(obj.get(key, "")) for key in EXPECTED_KEYS}


def explain_row(row: pd.Series, retries: int = 3) -> Dict[str, str]:
    user_prompt = build_user_prompt(row)
    for attempt in range(retries):
        try:
            raw = llm_client.generate(SYSTEM_PROMPT, user_prompt)
            return normalize_result(extract_json_object(raw))
        except ValueError as e:
            return {
                "explanation": f"Parse error: {str(e)}",
                "issue_type":  "parse_error",
                "validation":  "UNKNOWN",
                "confidence":  "low",
                "next_step":   "Manual review — LLM output could not be parsed",
            }
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(2 ** attempt)   # exponential backoff
            else:
                return {
                    "explanation": f"API error: {str(e)}",
                    "issue_type":  "api_error",
                    "validation":  "UNKNOWN",
                    "confidence":  "low",
                    "next_step":   "Retry failed — check API key and connection",
                }


llm_client = LLMClient(
    provider=PROVIDER,
    model_name=MODEL_NAME,
    api_key=API_KEY,
    base_url=BASE_URL,
    temperature=TEMPERATURE,
    timeout=TIMEOUT_SECONDS,
)

print("LLM client ready.")
print(f"Provider: {PROVIDER} | Model: {MODEL_NAME}")

LLM client ready.
Provider: openai | Model: gpt-4o-mini


## 6) Run LLM on flagged rows

Progress bar via `tqdm`. Results are renamed with `llm_` prefix and concatenated back.

In [7]:
results = []

for _, row in tqdm(work_df.iterrows(), total=len(work_df)):
    res = explain_row(row)
    results.append(res)
    time.sleep(SLEEP_SECONDS)

llm_df = pd.DataFrame(results).rename(columns={
    "explanation":        "llm_explanation",
    "issue_type":         "llm_issue_type",
    "validation":         "llm_validation",
    "validation_reasoning": "llm_validation_reasoning",
    "confidence":         "llm_confidence",
    "next_step":          "llm_next_step",
})

work_df = pd.concat([work_df.reset_index(drop=True), llm_df], axis=1)

print(f"\nDone. Shape: {work_df.shape}")
cols = [c for c in ['item_id', 'final_level', 'cr_profile_score', 'llm_validation', 'llm_confidence', 'llm_explanation'] if c in work_df.columns]
work_df[cols].head(3)

  0%|          | 0/490 [00:00<?, ?it/s]


Done. Shape: (490, 84)


,item_id,final_level,cr_profile_score,llm_validation,llm_confidence,llm_explanation
0,2020-2801-1-10074-1,HIGH,37.701743,TRUE_POSITIVE,medium,"This transaction is flagged due to a high trade pattern score and low CR profile score, indicating unusual trading b..."
1,2020-2701-5-10322-1,HIGH,41.482849,TRUE_POSITIVE,medium,"This transaction is flagged due to a high trade pattern score and low CR profile score, indicating unusual trading b..."
2,2020-2801-2-10978-1,LOW,37.453704,FALSE_POSITIVE,high,"The transaction shows a declared unit price of approximately 178.56 BHD for cotton t-shirts, which is significantly ..."


## 7) Validation summary

In [8]:
print("Validation breakdown:")
if 'llm_validation' in work_df.columns:
    print(work_df['llm_validation'].value_counts())
    
    print("\nValidation by flag level:")
    display(pd.crosstab(work_df['final_level'], work_df['llm_validation']))
    
    print("\nConfidence breakdown:")
    print(work_df['llm_confidence'].value_counts())
    
    print("\nIssue type breakdown:")
    print(work_df['llm_issue_type'].value_counts())
else:
    print("No LLM columns found — work_df may be empty or LLM did not run.")

Validation breakdown:
llm_validation
FALSE_POSITIVE    310
TRUE_POSITIVE     180
Name: count, dtype: int64

Validation by flag level:


llm_validation,FALSE_POSITIVE,TRUE_POSITIVE
final_level,,
HIGH,16,79
LOW,275,24
MEDIUM,19,77



Confidence breakdown:
llm_confidence
medium    348
high      142
Name: count, dtype: int64

Issue type breakdown:
llm_issue_type
benign_outlier          140
over_invoicing          139
unusual_partner         115
under_invoicing          55
unusual_route            19
cr_profile_deviation     10
other                     9
misclassification         3
Name: count, dtype: int64


## 8) Save output

In [9]:
work_df.to_csv(output_path, index=False)
print(f"Saved : {output_path}")
print(f"Shape : {work_df.shape}")
print(f"\nColumns:")
print(work_df.columns.tolist())


Saved : C:\Users\fawaz\Desktop\Project\outputs\LLM_Explainability.csv
Shape : (490, 84)

Columns:
['item_id', 'declaration_id', 'declaration_date', 'year', 'year_month', 'hs_code', 'hs_clean', 'hs6', 'hs2', 'hs_desc', 'commercial_description', 'active_cr', 'trade_type', 'regime', 'country_of_origin', 'country_of_export', 'country_of_destination', 'country_of_origin_code', 'country_of_export_code', 'country_of_destination_code', 'partner_country_code', 'uom', 'qty_by_uom', 'actual_unit_price', 'price_basis', 'local_amount_clean', 'invoice_amount_clean', 'sup_amount_clean', 'net_weight_clean', 'gross_weight_clean', 'log_actual_unit_price', 'median', 'mad', 'q1', 'q3', 'iqr', 'iqr_low', 'iqr_high', 'iqr_flag', 'robust_z', 'anomaly_score', 'price_level', 'partner_count', 'partner_probability', 'partner_rarity_raw', 'partner_rarity_score', 'route_count', 'route_probability', 'route_rarity_raw', 'route_rarity_score', 'active_months', 'month_mean', 'month_std', 'month_max', 'burstiness_raw', 